In [1]:
# ---------------------------------------------------------
# Verify pandas and PyArrow installation
# ---------------------------------------------------------
# PyArrow was installed after Jupyter had already started,
# so we verify that the restarted kernel can import it cleanly.

import pandas as pd
import pyarrow as pa

print("Pandas version:", pd.__version__)
print("PyArrow version:", pa.__version__)

Pandas version: 3.0.5
PyArrow version: 25.0.1


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Define project paths
# ---------------------------------------------------------
# The notebook is stored inside the `notebooks` directory,
# so its parent directory is the project root.

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Raw data directory exists:", RAW_DATA_DIR.exists())

Project root: C:\Users\saman\OneDrive\Desktop\Projects\retail-demand-intelligence
Raw data directory: C:\Users\saman\OneDrive\Desktop\Projects\retail-demand-intelligence\data\raw
Raw data directory exists: True


In [3]:
calendar = pd.read_csv(RAW_DATA_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DATA_DIR / "sell_prices.csv")
sales = pd.read_csv(RAW_DATA_DIR / "sales_train_evaluation.csv")

In [4]:
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)
print("Sales shape:", sales.shape)

Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1947)


In [5]:
print("\nCalendar columns:")
print(calendar.columns.tolist())

print("\nPrices columns:")
print(prices.columns.tolist())

print("\nFirst 15 Sales columns:")
print(sales.columns[:15].tolist())


Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Prices columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

First 15 Sales columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9']


In [6]:
sales.iloc[:, :15].head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0


In [7]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [8]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [9]:
print(f"Calendar: {calendar.shape}")
print(f"Prices:   {prices.shape}")
print(f"Sales:    {sales.shape}")

Calendar: (1969, 14)
Prices:   (6841121, 4)
Sales:    (30490, 1947)


In [10]:
print(f"Unique items:  {sales['item_id'].nunique():,}")
print(f"Unique stores: {sales['store_id'].nunique():,}")
print(f"Unique states: {sales['state_id'].nunique():,}")
print(f"Departments:   {sales['dept_id'].nunique():,}")
print(f"Categories:    {sales['cat_id'].nunique():,}")

Unique items:  3,049
Unique stores: 10
Unique states: 3
Departments:   7
Categories:    3


In [11]:
day_columns = [col for col in sales.columns if col.startswith("d_")]

print(f"Number of daily sales columns: {len(day_columns):,}")
print(f"First day: {day_columns[0]}")
print(f"Last day:  {day_columns[-1]}")

Number of daily sales columns: 1,941
First day: d_1
Last day:  d_1941


In [12]:
def memory_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print(f"Calendar memory: {memory_mb(calendar):,.2f} MB")
print(f"Prices memory:   {memory_mb(prices):,.2f} MB")
print(f"Sales memory:    {memory_mb(sales):,.2f} MB")

Calendar memory: 0.26 MB
Prices memory:   318.15 MB
Sales memory:    454.74 MB


In [13]:
print("States:")
print(sales["state_id"].value_counts())

print("\nStores:")
print(sales["store_id"].value_counts())

print("\nCategories:")
print(sales["cat_id"].value_counts())

print("\nDepartments:")
print(sales["dept_id"].value_counts())

States:
state_id
CA    12196
TX     9147
WI     9147
Name: count, dtype: int64

Stores:
store_id
CA_1    3049
CA_2    3049
CA_3    3049
CA_4    3049
TX_1    3049
TX_2    3049
TX_3    3049
WI_1    3049
WI_2    3049
WI_3    3049
Name: count, dtype: int64

Categories:
cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64

Departments:
dept_id
FOODS_3        8230
HOUSEHOLD_1    5320
HOUSEHOLD_2    5150
HOBBIES_1      4160
FOODS_2        3980
FOODS_1        2160
HOBBIES_2      1490
Name: count, dtype: int64


In [14]:
sales[["item_id", "dept_id", "cat_id", "store_id", "state_id"]].head(10)

,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA
5,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA
6,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA
7,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA
8,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA
9,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA


In [15]:
prices.info()
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 318.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), str(6)
memory usage: 454.7 MB


In [16]:
prices.dtypes

store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object

In [17]:
prices_optimized = prices.copy()

prices_optimized["store_id"] = prices_optimized["store_id"].astype("category")
prices_optimized["item_id"] = prices_optimized["item_id"].astype("category")

print(f"Before: {memory_mb(prices):,.2f} MB")
print(f"After:  {memory_mb(prices_optimized):,.2f} MB")

Before: 318.15 MB
After:  124.02 MB


In [18]:
sales_ca1 = sales[sales["store_id"] == "CA_1"].copy()
prices_ca1 = prices_optimized[prices_optimized["store_id"] == "CA_1"].copy()

print("Sales CA_1 shape:", sales_ca1.shape)
print("Prices CA_1 shape:", prices_ca1.shape)

print(f"Sales CA_1 memory:  {memory_mb(sales_ca1):,.2f} MB")
print(f"Prices CA_1 memory: {memory_mb(prices_ca1):,.2f} MB")

Sales CA_1 shape: (3049, 1947)
Prices CA_1 shape: (698412, 4)
Sales CA_1 memory:  45.48 MB
Prices CA_1 memory: 12.72 MB


In [19]:
categorical_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

for col in categorical_cols:
    sales_ca1[col] = sales_ca1[col].astype("category")

In [20]:
for col in day_columns:
    sales_ca1[col] = pd.to_numeric(
        sales_ca1[col],
        downcast="unsigned"
    )

In [21]:
print(f"Optimized Sales CA_1 memory: {memory_mb(sales_ca1):,.2f} MB")

Optimized Sales CA_1 memory: 5.92 MB


In [22]:
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

sales_long = sales_ca1.melt(
    id_vars=id_cols,
    value_vars=day_columns,
    var_name="d",
    value_name="sales"
)

In [23]:
print(f"Shape: {sales_long.shape}")
print(f"Memory: {memory_mb(sales_long):,.2f} MB")

sales_long.head(10)

Shape: (5918109, 8)
Memory: 133.11 MB


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
5,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
6,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [24]:
sales_long.dtypes

id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d                str
sales         uint16
dtype: object

In [25]:
sales_long["d"] = sales_long["d"].astype("category")

print(f"Optimized memory: {memory_mb(sales_long):,.2f} MB")
sales_long.dtypes

Optimized memory: 67.92 MB


id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d           category
sales         uint16
dtype: object

In [26]:
print("Sales rows:", f"{len(sales_long):,}")
print("Unique sales days:", sales_long["d"].nunique())
print("Calendar rows:", f"{len(calendar):,}")
print("Unique calendar d values:", calendar["d"].nunique())

Sales rows: 5,918,109
Unique sales days: 1941
Calendar rows: 1,969
Unique calendar d values: 1969


In [27]:
print("Duplicate calendar d values:", calendar["d"].duplicated().sum())

Duplicate calendar d values: 0


In [28]:
calendar_features = calendar[
    [
        "date",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "d",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
    ]
].copy()

In [29]:
calendar_features["date"] = pd.to_datetime(calendar_features["date"])

In [30]:
calendar_cat_cols = [
    "weekday",
    "d",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

for col in calendar_cat_cols:
    calendar_features[col] = calendar_features[col].astype("category")

In [31]:
# ---------------------------------------------------------
# Join daily sales with calendar information
# ---------------------------------------------------------
# The sales dataset identifies days using labels such as
# d_1, d_2, ..., d_1941 rather than actual calendar dates.
#
# calendar_features contains one record per `d` value, so
# this is a many-to-one relationship:
#
#     many sales observations -> one calendar day
#
# `validate="many_to_one"` ensures pandas raises an error
# if the calendar unexpectedly contains duplicate `d` keys.

sales_calendar = sales_long.merge(
    calendar_features,
    on="d",
    how="left",
    validate="many_to_one"
)

In [32]:
# Verify that the join did not accidentally duplicate or remove
# sales observations. A correct many-to-one join should preserve
# the original number of rows.

print(f"Before join: {len(sales_long):,}")
print(f"After join:  {len(sales_calendar):,}")
print(f"Shape:       {sales_calendar.shape}")
print(f"Memory:      {memory_mb(sales_calendar):,.2f} MB")

Before join: 5,918,109
After join:  5,918,109
Shape:       (5918109, 19)
Memory:      431.53 MB


In [33]:
# Check whether any sales observations failed to find a matching
# calendar date. We expect zero because every sales day from
# d_1 through d_1941 exists in the calendar dataset.

missing_dates = sales_calendar["date"].isna().sum()

print(f"Missing dates after join: {missing_dates:,}")

Missing dates after join: 0


In [34]:
# Display a small selection of useful columns to manually verify
# that the day identifier, actual date, calendar attributes,
# and sales target have been combined correctly.

sales_calendar[
    [
        "item_id",
        "store_id",
        "d",
        "date",
        "weekday",
        "sales",
        "wm_yr_wk",
        "event_name_1",
        "snap_CA",
    ]
].head(10)

,item_id,store_id,d,date,weekday,sales,wm_yr_wk,event_name_1,snap_CA
0,HOBBIES_1_001,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
1,HOBBIES_1_002,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
2,HOBBIES_1_003,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
3,HOBBIES_1_004,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
4,HOBBIES_1_005,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
5,HOBBIES_1_006,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
6,HOBBIES_1_007,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0
7,HOBBIES_1_008,CA_1,d_1,2011-01-29,Saturday,12,11101,NaN,0
8,HOBBIES_1_009,CA_1,d_1,2011-01-29,Saturday,2,11101,NaN,0
9,HOBBIES_1_010,CA_1,d_1,2011-01-29,Saturday,0,11101,NaN,0


In [35]:
# ---------------------------------------------------------
# Validate the composite key in the weekly price table
# ---------------------------------------------------------
# Each item/store/week combination should have at most one
# selling price. If duplicates exist, a merge could multiply
# rows and corrupt the modeling dataset.

price_key_cols = ["item_id", "store_id", "wm_yr_wk"]

duplicate_price_keys = prices_ca1.duplicated(
    subset=price_key_cols
).sum()

print(f"Duplicate price keys: {duplicate_price_keys:,}")

Duplicate price keys: 0


In [36]:
# Compare row count with the number of unique composite keys.
# These values should match when the key is truly unique.

unique_price_keys = prices_ca1[
    price_key_cols
].drop_duplicates().shape[0]

print(f"Price rows:        {len(prices_ca1):,}")
print(f"Unique price keys: {unique_price_keys:,}")

Price rows:        698,412
Unique price keys: 698,412


In [37]:
# ---------------------------------------------------------
# Check price coverage for sales observations
# ---------------------------------------------------------
# We compare the unique item/week combinations appearing in
# the sales data with the price table before performing the
# full multi-million-row merge.

sales_price_keys = sales_calendar[
    ["item_id", "store_id", "wm_yr_wk"]
].drop_duplicates()

price_coverage = sales_price_keys.merge(
    prices_ca1[
        ["item_id", "store_id", "wm_yr_wk", "sell_price"]
    ],
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left",
    validate="one_to_one"
)

missing_price_keys = price_coverage["sell_price"].isna().sum()

print(f"Unique sales price keys: {len(sales_price_keys):,}")
print(f"Missing price keys:      {missing_price_keys:,}")

Unique sales price keys: 847,622
Missing price keys:      161,406


In [38]:
# ---------------------------------------------------------
# Investigate missing price records
# ---------------------------------------------------------
# A missing weekly price may indicate that an item was not yet
# actively offered for sale during that week.
#
# Before deciding how to handle missing prices, we test whether
# those item/week combinations also have zero observed sales.

price_coverage_check = sales_calendar[
    ["item_id", "store_id", "wm_yr_wk", "sales"]
].merge(
    prices_ca1[
        ["item_id", "store_id", "wm_yr_wk", "sell_price"]
    ],
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

missing_price_rows = price_coverage_check[
    price_coverage_check["sell_price"].isna()
]

print(f"Rows with missing price: {len(missing_price_rows):,}")
print(
    f"Rows with missing price and zero sales: "
    f"{(missing_price_rows['sales'] == 0).sum():,}"
)
print(
    f"Rows with missing price and positive sales: "
    f"{(missing_price_rows['sales'] > 0).sum():,}"
)

Rows with missing price: 1,129,842
Rows with missing price and zero sales: 1,129,842
Rows with missing price and positive sales: 0


In [39]:
# Quantify how strongly missing price records are associated
# with zero demand.

zero_sales_pct = (
    (missing_price_rows["sales"] == 0).mean() * 100
)

positive_sales_pct = (
    (missing_price_rows["sales"] > 0).mean() * 100
)

print(f"Missing-price rows with zero sales:     {zero_sales_pct:.2f}%")
print(f"Missing-price rows with positive sales: {positive_sales_pct:.2f}%")

Missing-price rows with zero sales:     100.00%
Missing-price rows with positive sales: 0.00%


In [40]:
# ---------------------------------------------------------
# Join weekly selling prices onto daily sales observations
# ---------------------------------------------------------
# Prices are recorded weekly rather than daily. The calendar
# table supplied `wm_yr_wk`, which allows each daily sales
# observation to be matched to its item's weekly selling price.
#
# The price table has already been validated as unique on:
#     item_id + store_id + wm_yr_wk
#
# Therefore this is a many-to-one join.

model_data = sales_calendar.merge(
    prices_ca1[
        ["item_id", "store_id", "wm_yr_wk", "sell_price"]
    ],
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

In [41]:
# ---------------------------------------------------------
# Validate that the price join preserved the sales grain
# ---------------------------------------------------------
# Adding price information should not create or remove any
# daily sales observations.

print(f"Before price join: {len(sales_calendar):,}")
print(f"After price join:  {len(model_data):,}")
print(f"Shape:             {model_data.shape}")
print(f"Memory:            {memory_mb(model_data):,.2f} MB")

Before price join: 5,918,109
After price join:  5,918,109
Shape:             (5918109, 20)
Memory:            538.77 MB


In [42]:
# ---------------------------------------------------------
# Create product availability indicator
# ---------------------------------------------------------
# Our investigation showed that 100% of observations without
# a recorded selling price also had zero unit sales.
#
# Rather than imputing a fictitious price, preserve sell_price
# as missing and explicitly represent product availability.

model_data["is_available"] = (
    model_data["sell_price"].notna()
).astype("uint8")

In [43]:
# Check the distribution of available vs unavailable
# item-day observations.

availability_counts = model_data["is_available"].value_counts()

print(availability_counts)

is_available
1    4788267
0    1129842
Name: count, dtype: int64


In [44]:
# An unavailable product should never have positive observed
# sales based on the relationship identified above.

invalid_availability_rows = model_data[
    (model_data["is_available"] == 0)
    & (model_data["sales"] > 0)
]

print(
    "Unavailable observations with positive sales:",
    f"{len(invalid_availability_rows):,}"
)

Unavailable observations with positive sales: 0


In [45]:
# Inspect the first 20 daily observations for one product
# to verify the final table at the item-store-day grain.

example_item = "HOBBIES_1_008"

model_data.loc[
    model_data["item_id"] == example_item,
    [
        "date",
        "item_id",
        "sales",
        "sell_price",
        "is_available",
        "weekday",
        "event_name_1",
        "snap_CA",
    ]
].head(20)

,date,item_id,sales,sell_price,is_available,weekday,event_name_1,snap_CA
7,2011-01-29,HOBBIES_1_008,12,0.46,1,Saturday,NaN,0
3056,2011-01-30,HOBBIES_1_008,15,0.46,1,Sunday,NaN,0
6105,2011-01-31,HOBBIES_1_008,0,0.46,1,Monday,NaN,0
9154,2011-02-01,HOBBIES_1_008,0,0.46,1,Tuesday,NaN,1
12203,2011-02-02,HOBBIES_1_008,0,0.46,1,Wednesday,NaN,1
15252,2011-02-03,HOBBIES_1_008,4,0.46,1,Thursday,NaN,1
18301,2011-02-04,HOBBIES_1_008,6,0.46,1,Friday,NaN,1
21350,2011-02-05,HOBBIES_1_008,5,0.46,1,Saturday,NaN,1
24399,2011-02-06,HOBBIES_1_008,7,0.46,1,Sunday,SuperBowl,1
27448,2011-02-07,HOBBIES_1_008,0,0.46,1,Monday,NaN,1


## Save Processed Dataset

The validated CA_1 analytical dataset is saved as a Parquet file so that
subsequent analysis and modeling notebooks can load the prepared data
directly without repeating the raw-data transformation and join pipeline.

In [46]:
# ---------------------------------------------------------
# Save the processed CA_1 analytical dataset
# ---------------------------------------------------------
# Parquet is used instead of CSV because it:
#   1. preserves column data types,
#   2. supports compression,
#   3. loads faster for large analytical datasets, and
#   4. generally requires less disk space than CSV.
#
# This creates a reusable checkpoint for downstream EDA,
# feature engineering, and modeling notebooks.

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Ensure the processed-data directory exists.
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

processed_file = PROCESSED_DATA_DIR / "ca1_model_data.parquet"

# Save without a DataFrame index because the index is not
# part of the analytical dataset.
model_data.to_parquet(
    processed_file,
    index=False,
    engine="pyarrow"
)

print("Processed dataset saved to:")
print(processed_file)

Processed dataset saved to:
C:\Users\saman\OneDrive\Desktop\Projects\retail-demand-intelligence\data\processed\ca1_model_data.parquet


In [47]:
# ---------------------------------------------------------
# Validate the processed Parquet checkpoint
# ---------------------------------------------------------
# Reload the saved dataset from disk to verify that:
#   1. the Parquet file can be read successfully, and
#   2. the number of rows and columns matches model_data.
#
# This ensures the saved checkpoint is reliable before it is
# used by downstream EDA and modeling notebooks.

model_data_check = pd.read_parquet(
    processed_file,
    engine="pyarrow"
)

print("Original shape:", model_data.shape)
print("Reloaded shape:", model_data_check.shape)

# Stop execution if the saved dataset has a different shape.
assert model_data_check.shape == model_data.shape, (
    "Processed dataset shape does not match the original dataset."
)

print("Validation successful: shapes match.")

Original shape: (5918109, 21)
Reloaded shape: (5918109, 21)
Validation successful: shapes match.


In [48]:
# ---------------------------------------------------------
# Compare disk size with in-memory size
# ---------------------------------------------------------
# Parquet uses columnar storage and compression, so the file
# stored on disk should generally be considerably smaller than
# the DataFrame's memory footprint.

file_size_mb = processed_file.stat().st_size / (1024 ** 2)

memory_size_mb = (
    model_data.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"Parquet file size: {file_size_mb:.2f} MB")
print(f"DataFrame memory size: {memory_size_mb:.2f} MB")

Parquet file size: 25.81 MB
DataFrame memory size: 544.48 MB
